In [1]:
import seaborn as sns   
import matplotlib.pyplot as plt 
import numpy as np
import pandas as pd
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer



In [2]:
df = pd.read_parquet('../data_sample/datos_licitaciones.parquet', engine='fastparquet')


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 37700 entries, 572837 to 2087811
Data columns (total 35 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   Situació contractual                 37700 non-null  object 
 1   Exercici                             37700 non-null  int64  
 2   Àmbit organitzatiu                   37700 non-null  object 
 3   Identificador agrupació organisme    37700 non-null  object 
 4   Agrupació organisme                  37700 non-null  object 
 5   Identificador organisme contractant  37700 non-null  object 
 6   Organisme contractant                37700 non-null  object 
 7   Codi de l’expedient                  37700 non-null  object 
 8   Procediment d’adjudicació            37700 non-null  object 
 9   Tipus de contracte                   37700 non-null  object 
 10  Descripció de l’expedient            37700 non-null  object 
 11  Número de lot             

In [4]:
df

,Situació contractual,Exercici,Àmbit organitzatiu,Identificador agrupació organisme,Agrupació organisme,Identificador organisme contractant,Organisme contractant,Codi de l’expedient,Procediment d’adjudicació,Tipus de contracte,...,Tipus de modificació,Import de la modificació,Data aprovació modificació,Termini modificació anys,Termini modificació mesos,Termini modificació dies,Tipus de liquidació,Data de liquidació,Causa de resolució,Import de la liquidació
572837,liquidació,2022,Entitats de l'Administració Local,4316280001,Ajuntament de Vandellòs i l'Hospitalet de l'In...,4316280001,Ajuntament de Vandellòs i l'Hospitalet de l'In...,0225 / 2022,Menor,3. SUBMINISTRAMENTS,...,,NaN,,NaN,NaN,NaN,COMPLIMENT,08/02/2022,,2.38
609033,liquidació,2023,Departaments i Sector Públic de la Generalitat...,1500,DEPARTAMENT DE SALUT,1542,Institut Català de la Salut (ICS) Barcelonès H...,1101369832,Menor,5. SERVEIS,...,,NaN,,NaN,NaN,NaN,COMPLIMENT,14/11/2023,,170.68
1612984,liquidació,2024,Entitats de l'Administració Local,0822050006,Ajuntament de Sant Julià de Vilatorta,0822050006,Ajuntament de Sant Julià de Vilatorta,SU41,Menor,3. SUBMINISTRAMENTS,...,,NaN,,NaN,NaN,NaN,COMPLIMENT,13/05/2024,,45.81
1412433,liquidació,2021,Departaments i Sector Públic de la Generalitat...,1400,DEPARTAMENT DE CULTURA,7915100002,Biblioteca de Catalunya,BC-2021-135,Menor,5. SERVEIS,...,,NaN,,NaN,NaN,NaN,COMPLIMENT,15/06/2021,,2150.13
951405,liquidació,2025,Entitats de l'Administració Local,8000840003,Diputació de Barcelona,8000840003,Diputació de Barcelona,2025/4118 - 9,Menor,5. SERVEIS,...,,NaN,,NaN,NaN,NaN,COMPLIMENT,25/03/2025,,880.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3383863,liquidació,2023,Departaments i Sector Públic de la Generalitat...,1500,DEPARTAMENT DE SALUT,7996100024,Fundació IDIAP Jordi Gol i Gurina,2023/R_107,Menor,5. SERVEIS,...,,NaN,,NaN,NaN,NaN,COMPLIMENT,24/02/2023,,926.73
3122031,liquidació,2023,Entitats de l'Administració Local,1711430008,Ajuntament d'Olot,1711430008,Ajuntament d'Olot,CC012023000848,Menor,5. SERVEIS,...,,NaN,,NaN,NaN,NaN,COMPLIMENT,13/12/2023,,2500.00
688482,liquidació,2021,Departaments i Sector Públic de la Generalitat...,9610930008,"DEPARTAMENT DE TERRITORI, HABITATGE I TRANSICI...",9805180001,Institut Metròpoli,F / 2021 / 339,Menor,5. SERVEIS,...,,NaN,,NaN,NaN,NaN,COMPLIMENT,15/12/2021,,220.00
3290182,liquidació,2025,Departaments i Sector Públic de la Generalitat...,9610930008,"DEPARTAMENT DE TERRITORI, HABITATGE I TRANSICI...",9610930818,Ferrocarrils de la Generalitat de Catalunya,CONTM/2025/0000001230,Menor,5. SERVEIS,...,,NaN,,NaN,NaN,NaN,COMPLIMENT,03/08/2025,,6321.47


In [5]:
df = df.loc[df["Import de la liquidació"] > 0].copy()


In [6]:
'''
Podemos observar que todas las filas con el mismo tipo de situación, no nos aporta ningún tipo de información, por tanto, decidimos eliminarla

'''
df.iloc[:, 0].value_counts()
df.drop("Situació contractual", axis = 1, inplace = True)

In [7]:
'''
Si bien se trata ya de una feature numerica, para mejorar el rendimiento del modelo, aplicaremos una transformación, para que el modelo no interprete siempre estos valores
como año, dado que el modelo no entiende lo que son, solo que son modelos 

'''
df.iloc[:, 0].value_counts().sort_values()

clave_Exercici = {año: i for i, año in enumerate(range(2007, 2031))}

df["Exercici"] = df["Exercici"].map(clave_Exercici)

In [8]:
clave_Ambit_organitzatiu = {
    "Universitats": 0,
    "Departaments i Sector Públic de la Generalitat de Catalunya": 1,
    "Entitats de l'Administració Local": 2
}

df["Àmbit organitzatiu"] = df["Àmbit organitzatiu"].map(clave_Ambit_organitzatiu)

In [9]:
'''
Hemos descubierto que el campo "Identificador agrupació organisme" no se trata de un numero aleatorio, en realidad, tiene su propia nomenclatura, los primeros dos digitos
hacen referencia a la dirección postal del lugar de la licitación, por lo que podemos añadir una columna adicional a partir de esta, mas explicativa que el organismo contratante,
para identificar el lugar de la licitacion. En concreto, los codigos 08,17,25,43 hacen referencia a las direcciones postales de Barcelona, Girona, LLeida y Tarragona.
El resto de valores que no entran en estas caracteristicas, se tratan de organizaciones que no comprende este indice, y se hara una transformacion a parte
'''
df["Indice_Identificador_agrupació _organisme"] = df["Identificador agrupació organisme"].str[:2]
df["Indice_Identificador_agrupació _organisme"].value_counts()

clave_referencia_licitacio = {
    "08": "Barcelona",
    "80": "Barcelona",
    "96": "Departamentos_Estudio",
    "15": "Salud",
    "43": "Tarragona",
    "17": "Girona",
    "14": "Cultura",
    "25": "Lleida",
    "00": "Sociales",
    "81": "Consejo_Comarcal",
    "79": "Fundacion_privada",
    "98": "Desarrollo",
    "99": "Varios",
    "70": "Varios",
    "10": "Varios",
    "90": "Varios",
    "82": "Varios",
    "95": "Varios"

}

df["Referencia_Licitació"] = df["Indice_Identificador_agrupació _organisme"].map(clave_referencia_licitacio)
df["Referencia_Licitació"] = np.where(
    df["Identificador agrupació organisme"] =="8000",
    "Universidades",
    df["Referencia_Licitació"]
)

'''
Y ahora hacemos la transformación pertinente, para convertir la feature en numerica 
'''

clave_referencia_licitacio_numerico = {
    "Barcelona": 0,
    "Departamentos_Estudio": 1,
    "Salud": 2,
    "Tarragona": 3,
    "Universidades": 4,
    "Girona": 5,
    "Cultura": 6,
    "Lleida": 7,
    "Sociales": 8,
    "Consejo_Comarcal": 9,
    "Fundacion_privada": 10,
    "Desarrollo": 11,
    "Varios": 12,  
}
df["Referencia_Licitació"] = df["Referencia_Licitació"].map(clave_referencia_licitacio_numerico)

'''
Una vez resuelto esto, toca eliminar la columna del identificador, dada su alta cardinalidad
'''
df.drop("Identificador agrupació organisme", axis = 1, inplace = True)

In [10]:
'''
Tambien eliminaremos la columna agrupació organisme, identificador organisme, i organisme contractant, dada la transformación anterior que nos permite generalizar 
mucho mejor estos datos
'''
df.drop("Agrupació organisme", axis = 1, inplace = True)
df.drop("Identificador organisme contractant", axis = 1, inplace = True)
df.drop("Organisme contractant", axis = 1, inplace = True)

In [11]:
"""
Hemos intentado buscar documentacion acerca de la variable "Codi de l"expedient". Si bien da información, el problema es que se trata de una feature de texto libre, y esulta complicado
encontrar un patron para poder deducir la información, y poder hacer algo con ella. Afortunadamente, otras columnas del dataset ya son explicativas de las caracteristicas de 
dicho codigo, y por tanto, nos podemos permitir perder dicha columna
"""

df.drop("Codi de l’expedient", axis = 1, inplace = True)


In [12]:
'''
Vemos que tiene una estructura muy irregular, podemos transformar todos los valores menores en 0, y el resto ponerlo ponerlos en 1 
'''
df.iloc[:, 2].value_counts().sort_values()

df["Procediment d’adjudicació"] = np.where(
    df["Procediment d’adjudicació"] =="Menor",
    0,
    1
)


In [13]:
'''
Nuevamente una columna categorica, que podemos aplicar un mapeado para facilitar el proceso del modelo 
'''

clave_tipus_contracte = {
    "5. SERVEIS": 0,
    "3. SUBMINISTRAMENTS": 1,
    "1. OBRES": 2,
    "GESTIÓ DE SERVEI PÚBLIC": 3,
    "SERVEIS": 4,
    "10. PRIVAT D'ADMINISTRACIO PUBLICA": 5,
    "SUBMINISTRAMENTS": 6,
    "8. CONCESSIÓ DE SERVEIS": 7,
    "2. GESTIÓ DE SERVEI PÚBLIC": 8,
    "6. ADMINISTRATIU ESPECIAL": 9,
    "ADMINISTRATIU ESPECIAL": 10,
    "OBRES": 11,
    "COL·LABORACIÓ PÚBLIC- PRIVAT": 12,
    "CONCESSIÓ D'OBRA PÚBLICA": 13,
    "9. CONCESSIÓ D'OBRES": 14,
    "CONSULTORIA I ASSISTÈNCIA (antic)": 15
}

df["Tipus de contracte"] = df["Tipus de contracte"].map(clave_tipus_contracte)

In [14]:


df.iloc[:, 4].value_counts().sort_values(ascending= False)


Descripció de l’expedient
COMPRA DE FARMACIA                                           1206
Espectacle - PROGRAMA.CAT                                     280
Materials de Laboratori Científic (Varis:Óptica/Fotònica)     233
COMPRES/SERVEIS MENORS                                         87
REACTIUS                                                       84
                                                             ... 
Male BALB/cByJ MOUSE-AGE Range Cohort-Dr. Melgar                1
Pressupost n.3954 Cladellas. Subministrament joguines.          1
Ajustar porta corredissa entrada sud                            1
Mini ordenador Barebone Intel Nuc I3 8GB Ram 480GB              1
CRISTAL DCHO MTO                                                1
Name: count, Length: 31668, dtype: int64

In [15]:
'''
Esta columna esta perfecta tal y como viene, y no es neceario aplicar transformaciones.
'''

df.iloc[:, 5].value_counts().sort_values(ascending= False)


Número de lot
1     37551
2        37
3        15
4        12
5         6
8         4
7         4
6         3
15        2
16        2
9         2
10        2
17        1
35        1
27        1
13        1
53        1
11        1
18        1
32        1
24        1
59        1
25        1
47        1
37        1
Name: count, dtype: int64

In [16]:
'''
El codigo cpv contiene información muy interesante acerca del ambito al que se pueda referir la licitacion, para ello necesitamos hacer una transformacion de la columna, y 
procesar los datos
'''
df.iloc[:, 6].value_counts().sort_values(ascending= False)
df["Codi_CPV_def"] = df["Codi CPV"].str[:8]

with open('Codigos_CPV.txt', 'r', encoding= "utf-8") as f:
    lineas = f.readlines()

datos = [linea.strip().split('-', 1) for linea in lineas]

df_CPV = pd.DataFrame(datos, columns=['Codi_CPV', 'Descripcio_CPV'])

In [17]:
df_CPV

,Codi_CPV,Descripcio_CPV
0,03000000,"Productos de la agricultura, ganadería, pesca..."
1,03100000,Productos de la agricultura y horticultura
2,03110000,"Cultivos, productos comerciales de jardinería..."
3,03111000,Semillas
4,03111100,Soja
...,...,...
9449,98513300,Personal temporal para particulares
9450,98513310,Servicios de ayuda en tareas domésticas
9451,98514000,Servicios domésticos
9452,98900000,Servicios prestados por organizaciones y enti...


In [18]:
df['Codi_CPV_def'] = df['Codi_CPV_def'].astype(str).str.strip()
df_CPV['Codi_CPV'] = df_CPV['Codi_CPV'].astype(str).str.strip()

In [19]:
df = df.merge(df_CPV[['Codi_CPV', 'Descripcio_CPV']], left_on='Codi_CPV_def', right_on='Codi_CPV', how='left')


In [ ]:
'''
Una vez hecho el procesado de esta columna, podemos eliminar las columnas de información relativa para evitar
ruido en el modelo. Por desgracia, esta columna no estaba complimentada originalmente, por lo que debemos
generalizar en los nulos, poniendo "Desconocido", en cada uno de ellos
'''

df.drop(columns=['Codi CPV', 'Codi_CPV_def', 'Codi_CPV'], inplace=True)


In [27]:
df

,Exercici,Àmbit organitzatiu,Procediment d’adjudicació,Tipus de contracte,Descripció de l’expedient,Número de lot,Adjudicatari,Import d’adjudicació,Data d’adjudicació,Descripció del lot,...,Termini modificació anys,Termini modificació mesos,Termini modificació dies,Tipus de liquidació,Data de liquidació,Causa de resolució,Import de la liquidació,Indice_Identificador_agrupació _organisme,Referencia_Licitació,Descripcio_CPV
0,15,2,0,1,MOSQUETONS,1,"EL ALMECEN FERRETERIA, S.L.",2.38,08/02/2022,MOSQUETONS,...,NaN,NaN,NaN,COMPLIMENT,08/02/2022,,2.38,43,3,NaN
1,16,1,0,0,CRISTAL DCHO MTO,1,"ALUMINI I VIDRE VALLMAR, SLL",170.68,14/11/2023,CRISTAL DCHO MTO,...,NaN,NaN,NaN,COMPLIMENT,14/11/2023,,170.68,15,2,Servicios de reparación y mantenimiento
2,17,2,0,1,AOC:172033646 Formatge manxego ventero / Pit d...,1,CASA GALLART ARTESANS SL,45.81,13/05/2024,AOC:172033646 Formatge manxego ventero / Pit d...,...,NaN,NaN,NaN,COMPLIMENT,13/05/2024,,45.81,08,0,NaN
3,14,1,0,0,"Producció gràfica, muntatge i desmuntatge exp ...",1,ESTUDIOS DURERO,2150.13,15/04/2021,"Producció gràfica, muntatge i desmuntatge exp ...",...,NaN,NaN,NaN,COMPLIMENT,15/06/2021,,2150.13,14,6,Servicios de organización de ferias y exposic...
4,18,2,0,0,Contractació de l'edició EXT2025/0445 de l'acc...,1,"SERRA AUFERIL, CRISTINA",880.00,25/03/2025,Contractació de l'edició EXT2025/0445 de l'acc...,...,NaN,NaN,NaN,COMPLIMENT,25/03/2025,,880.00,80,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37648,16,1,0,0,Formació i Docència,1,Fundació Universitat Oberta de Catalunya,926.73,24/02/2023,Formació i Docència,...,NaN,NaN,NaN,COMPLIMENT,24/02/2023,,926.73,15,2,NaN
37649,16,2,0,0,CULTURA - Contractar diferents espectacles esc...,1,"Buenritmo Producciones, SL",2500.00,13/12/2023,CULTURA - Contractar diferents espectacles esc...,...,NaN,NaN,NaN,COMPLIMENT,13/12/2023,,2500.00,17,5,NaN
37650,14,1,0,0,OHB_IIAB: Quata mensual manteniment informàtic...,1,"SOSMATIC, SLU",220.00,15/12/2021,OHB_IIAB: Quata mensual manteniment informàtic...,...,NaN,NaN,NaN,COMPLIMENT,15/12/2021,,220.00,96,1,NaN
37651,18,1,0,0,Servei treball pintura taller manteniment Ribes,1,JORGE SARDÁ REÑE,6321.47,04/07/2025,Servei treball pintura taller manteniment Ribes,...,NaN,NaN,NaN,COMPLIMENT,03/08/2025,,6321.47,96,1,Reparación y mantenimiento de aseos públicos


In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37653 entries, 0 to 37652
Data columns (total 34 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   Exercici                                   37653 non-null  int64  
 1   Àmbit organitzatiu                         37653 non-null  int64  
 2   Procediment d’adjudicació                  37653 non-null  int64  
 3   Tipus de contracte                         37653 non-null  int64  
 4   Descripció de l’expedient                  37653 non-null  object 
 5   Número de lot                              37653 non-null  int64  
 6   Codi CPV                                   37653 non-null  object 
 7   Adjudicatari                               37653 non-null  object 
 8   Import d’adjudicació                       37653 non-null  float64
 9   Data d’adjudicació                         37653 non-null  object 
 10  Descripció del lot    